# 🧠 Chunking en RAG — versión 100% compatible

Este notebook evita por completo los imports que están fallando en tu entorno.

## Qué vas a ver
- qué es un chunk,
- por qué dividir un documento,
- cómo cambia según el tamaño,
- cómo funciona el **overlap**,
- y una mini búsqueda semántica local.

> Esta versión no usa `langchain_text_splitters` ni `sentence_transformers`,
> porque en tu entorno actual esos imports están rompiéndose por una incompatibilidad interna.


## 1. Texto de ejemplo

In [ ]:
text = """Nuestra política de reembolso aplica únicamente si el producto presenta defectos de fábrica dentro de los primeros 30 días.
El cliente debe presentar comprobante de compra.
No aplican reembolsos por mal uso del producto.
Los tiempos de procesamiento pueden variar entre 5 y 10 días hábiles.
El soporte está disponible de lunes a viernes de 9am a 6pm.
Los cambios solo aplican si el producto está en su empaque original.
Las devoluciones no cubren daños ocasionados por el cliente.
En compras internacionales, los tiempos de revisión pueden ser mayores.
Para solicitar una revisión, el cliente debe contactar al equipo de soporte."""

print(text)

## 2. Medir tamaño del documento

In [ ]:
import pandas as pd

stats = pd.DataFrame([{
    "caracteres": len(text),
    "palabras_aprox": len(text.split()),
    "lineas": len(text.splitlines())
}])

stats

## 3. Chunking manual simple

In [ ]:
def split_text(text, size=120):
    chunks = []
    for i in range(0, len(text), size):
        chunks.append(text[i:i+size])
    return chunks

manual_chunks = split_text(text, size=120)

for i, chunk in enumerate(manual_chunks, start=1):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---\n{chunk}")

## 4. Comparar distintos tamaños

In [ ]:
sizes = [60, 100, 150, 220]
rows = []

for size in sizes:
    chunks = split_text(text, size=size)
    rows.append({
        "chunk_size": size,
        "total_chunks": len(chunks),
        "avg_chunk_length": round(sum(len(c) for c in chunks) / len(chunks), 2),
        "max_chunk_length": max(len(c) for c in chunks)
    })

df_sizes = pd.DataFrame(rows)
df_sizes

## 5. Visualización con Plotly

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    data=[go.Bar(x=df_sizes["chunk_size"], y=df_sizes["total_chunks"])]
)

fig.update_layout(
    title="Cantidad de chunks según chunk_size",
    xaxis_title="chunk_size",
    yaxis_title="total_chunks"
)

fig.show()

## 6. Problema: cortar mal una idea

Si el chunk es demasiado pequeño, una idea importante puede quedar partida entre dos bloques.


In [ ]:
bad_chunks = split_text(text, size=80)

for i, chunk in enumerate(bad_chunks, start=1):
    print(f"\n--- Chunk {i} ---\n{chunk}")

## 7. Overlap manual

In [ ]:
def split_text_with_overlap(text, chunk_size=120, overlap=30):
    if overlap >= chunk_size:
        raise ValueError("overlap debe ser menor que chunk_size")

    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += step

    return chunks

overlap_chunks = split_text_with_overlap(text, chunk_size=120, overlap=30)

for i, chunk in enumerate(overlap_chunks, start=1):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---\n{chunk}")

## 8. Comparar sin overlap vs con overlap

In [ ]:
comparison = pd.DataFrame([
    {
        "tipo": "sin_overlap",
        "total_chunks": len(manual_chunks),
        "primer_chunk_len": len(manual_chunks[0]),
    },
    {
        "tipo": "con_overlap",
        "total_chunks": len(overlap_chunks),
        "primer_chunk_len": len(overlap_chunks[0]),
    }
])

comparison

## 9. Mini búsqueda semántica local

Para mantener compatibilidad total, usamos:

- `TfidfVectorizer`
- `cosine_similarity`

No es un embedding moderno de producción, pero es perfecto para explicar el concepto del reel.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "¿Cuál es la política de reembolso?"

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(overlap_chunks + [query])

chunk_vectors = X[:-1]
query_vector = X[-1]

scores = cosine_similarity(query_vector, chunk_vectors).flatten()
best_idx = int(np.argmax(scores))

print("Consulta:", query)
print("\nScores:", scores)
print("\nChunk más relevante:")
print(overlap_chunks[best_idx])

## 10. Top-K resultados

In [ ]:
k = 3
top_indices = scores.argsort()[::-1][:k]

results = []
for rank, idx in enumerate(top_indices, start=1):
    results.append({
        "rank": rank,
        "score": round(float(scores[idx]), 4),
        "chunk_preview": overlap_chunks[idx][:100] + ("..." if len(overlap_chunks[idx]) > 100 else "")
    })

pd.DataFrame(results)

## 11. Conclusión

In [ ]:
summary = [
    "1. Un documento grande no siempre se debe enviar completo.",
    "2. Por eso lo dividimos en chunks.",
    "3. Si cortas mal, rompes significado.",
    "4. El overlap ayuda a conservar contexto.",
    "5. Luego recuperamos solo los chunks más relevantes."
]

for line in summary:
    print(line)